# 03. Test и итоговые сравнения
T4 для генерации. Используй те же GIT_REF, RUN_NAME и TRAIT. Слой и alpha уже выбраны на dev.


In [ ]:
from pathlib import Path
import subprocess

REPO_URL = "https://github.com/danilov-e/lora-activation-steering.git"
GIT_REF = "main"  # Ветка или тег, куда загружен этот проект.
PROJECT = Path("/content/lora-activation-steering")

if not PROJECT.exists():
    subprocess.run(
        ["git", "clone", "--branch", GIT_REF, REPO_URL, str(PROJECT)], check=True
    )
print("Commit:", subprocess.check_output(
    ["git", "-C", str(PROJECT), "rev-parse", "--short", "HEAD"], text=True
).strip())
%pip -q install -r {PROJECT}/requirements.txt


## 1. Открыть запуск
При двух настройках и трёх повторах: 18 условий × 100 вопросов × 3 = 5400 ответов на признак; 1080 запросов судье по пять ответов. Повторы и выбор настроек берутся из сохранённого запуска.


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import sys
sys.dont_write_bytecode = True
sys.path.insert(0, str(PROJECT))
from src import workflow as w
from src import judge as j
import pandas as pd
from IPython.display import display, Image

RUNS = Path("/content/drive/MyDrive/lora_activation_steering/runs")
RUN_NAME = "research_01"
TRAIT = "sycophancy"  # sycophancy / evil / hallucination

RUN = w.open_run(PROJECT, RUN_NAME, TRAIT, runs_dir=RUNS)
info, _ = w.run_info(RUN)
choices = w.read_json(RUN/"choices.json")["choices"]
expected = (2 + 8*len(choices)) * len(w.pairs(RUN, "test")) * info["config"]["test_repeats"]
print("Ответов:", expected, "| Запросов по пять:", (expected + 4)//5)
display(pd.DataFrame(choices))


## 2. Генерация
| Модель | Без стиринга | Старый вектор | Новый raw | Новый с нормой старого |
|---|---|---|---|---|
| Базовая | да | да | — | — |
| normal LoRA | да | да | да | да |
| steered LoRA | да | да | да | да |

Для новых векторов используются прежние слой и alpha. Norm-matched меняет только длину. Соответствующие вопросы и повторы имеют одинаковые seed. Прогресс сохраняется после каждого ответа.


In [ ]:
TEST = w.generate_test(RUN)


## 3. OpenAI API
Тот же судья, параметры и рубрики, что на dev. Только целевой признак и coherence; условия скрыты. Следующая ячейка выполняет платные запросы и продолжает с сохранённого пакета.


In [ ]:
import os
from getpass import getpass
from google.colab import userdata

if not os.environ.get("OPENAI_API_KEY"):
    try:
        key = userdata.get("OPENAI_API_KEY")
    except Exception:
        key = None
    os.environ["OPENAI_API_KEY"] = key or getpass("OpenAI API key: ")

judge_config = w.read_json(RUN/"dev"/"judge"/"manifest.json")["identity"]
JUDGED_OUTPUT = RUN/"test"/"all"
display(j.judge(
    JUDGED_OUTPUT, TRAIT,
    model=judge_config["model"], reasoning=judge_config["reasoning"],
    batch_size=judge_config["batch_size"],
    max_output_tokens=judge_config["max_output_tokens"],
))


## 4. Итоговые таблицы и графики
within_model_steering — эффект включения вектора относительно той же модели без стиринга.

restoration_gap — отличие от исходной модели со старым стирингом; близость к нулю сама по себе не доказывает эквивалентность.

change_in_steering_effect — насколько изменился эффект включения стиринга после LoRA. Интервалы строятся по вопросам, повторы сначала усредняются.


In [ ]:
summary, effects, figures = w.test_report(RUN)
display(summary.round(2))
display(effects.round(2))
for figure in figures:
    display(Image(filename=str(figure)))


## 5. Примеры изменений
Крупнейшее по модулю и медианное изменение относительно baseline. Все ответы и оценки сохранены в report/answers_and_scores.csv.


In [ ]:
for choice in choices:
    condition = "steered_" + choice["name"] + ":" + choice["name"] + "_old"
    for _, row in w.examples(JUDGED_OUTPUT, TRAIT, condition).iterrows():
        print("\n", row.question_id, row.condition, row[TRAIT], row.coherence)
        print(row.question)
        print(row.answer)
